# La legge dentro la loss

Il codice del capitolo [«La legge dentro la loss»](https://book.paithon.it/main/PINN/come-funziona.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## La legge dentro la loss

[Leggi la pagina](https://book.paithon.it/main/PINN/come-funziona.html)


### La PINN, riga per riga


In [ ]:
import numpy as np
import torch
from torch import nn

torch.manual_seed(42)

# Parametri fisici della molla: massa, smorzamento, rigidezza
m, c, k = 1.0, 0.4, 4.0

# La candidata soluzione: un MLP che da t produce u(t)
rete = nn.Sequential(
    nn.Linear(1, 32), nn.Tanh(),
    nn.Linear(32, 32), nn.Tanh(),
    nn.Linear(32, 32), nn.Tanh(),
    nn.Linear(32, 1),
)

# Punti di collocazione: 200 istanti a caso in [0, 10]
t_c = 10.0 * torch.rand(200, 1)     # shape (200, 1)
t_c.requires_grad_(True)            # derivate RISPETTO ALL'INPUT

# L'istante iniziale, dove imporremo u(0)=1 e u'(0)=0
t_0 = torch.zeros(1, 1, requires_grad=True)

ottimizzatore = torch.optim.Adam(rete.parameters(), lr=1e-3)

In [ ]:
for epoca in range(30_000):
    ottimizzatore.zero_grad()

    # 1) fisica: residuo m*u'' + c*u' + k*u sui punti di collocazione
    u = rete(t_c)                                        # shape (200, 1)
    u_t = torch.autograd.grad(u, t_c, torch.ones_like(u),
                              create_graph=True)[0]      # u'(t)
    u_tt = torch.autograd.grad(u_t, t_c, torch.ones_like(u_t),
                               create_graph=True)[0]     # u''(t)
    residuo = m * u_tt + c * u_t + k * u
    loss_fisica = (residuo ** 2).mean()

    # 2) condizioni iniziali: u(0) = 1 e u'(0) = 0
    u_0 = rete(t_0)
    u_t0 = torch.autograd.grad(u_0, t_0, torch.ones_like(u_0),
                               create_graph=True)[0]
    loss_iniziale = (u_0 - 1.0).pow(2).mean() + u_t0.pow(2).mean()

    # 3) loss totale, con piu' peso all'unico ancoraggio che abbiamo
    loss = loss_fisica + 100.0 * loss_iniziale
    loss.backward()
    ottimizzatore.step()

    if epoca % 5_000 == 0:
        print(f"epoca {epoca:6d} | loss {loss.item():.2e}")

In [ ]:
# La soluzione analitica, per dare i voti alla rete
gamma = c / (2 * m)                        # 0.2
omega_d = np.sqrt(k / m - gamma ** 2)      # sqrt(3.96) ~ 1.98997

t_test = np.linspace(0.0, 10.0, 500)
u_esatta = np.exp(-gamma * t_test) * (
    np.cos(omega_d * t_test) + (gamma / omega_d) * np.sin(omega_d * t_test)
)

def diagnosi(rete, t_controllo):
    """Tre misure che conviene tenere separate: il residuo DOVE la rete e'
    stata controllata, il residuo su una griglia fitta che non ha mai visto,
    e l'errore vero contro la formula esatta."""
    def residuo_su(t):
        u = rete(t)
        u_t = torch.autograd.grad(u, t, torch.ones_like(u),
                                  create_graph=True)[0]
        u_tt = torch.autograd.grad(u_t, t, torch.ones_like(u_t))[0]
        return ((m * u_tt + c * u_t + k * u) ** 2).mean().item()

    t_griglia = torch.tensor(t_test, dtype=torch.float32).reshape(-1, 1)
    t_griglia.requires_grad_(True)
    with torch.no_grad():
        errore = np.abs(rete(t_griglia).squeeze().numpy() - u_esatta)
    return residuo_su(t_controllo), residuo_su(t_griglia), errore


res_punti, res_griglia, errore = diagnosi(rete, t_c)
print(f"residuo sui 200 punti di collocazione: {res_punti:.2e}")
print(f"residuo su una griglia fitta         : {res_griglia:.2e}")
print(f"errore massimo                       : {errore.max():.3f}")
print(f"  sui primi 5 secondi                : {errore[t_test <= 5.0].max():.3f}")
print(f"  sugli ultimi 5 secondi             : {errore[t_test > 5.0].max():.3f}")

# La promessa si verifica sul posto.
assert errore.max() < 0.45, (
    f"errore massimo {errore.max():.3f}: la rete non sta ricostruendo "
    "l'oscillazione, e' collassata sulla curva piatta ferma sullo zero."
)

### Lo stesso codice, un altro seme


In [ ]:
def addestra(seme, epoche=30_000, peso=100.0, verboso=True):
    """Come l'addestramento di sopra: cambia solo il punto di partenza
    (e, se lo si chiede, il moltiplicatore delle condizioni iniziali)."""
    torch.manual_seed(seme)
    rete = nn.Sequential(
        nn.Linear(1, 32), nn.Tanh(),
        nn.Linear(32, 32), nn.Tanh(),
        nn.Linear(32, 32), nn.Tanh(),
        nn.Linear(32, 1),
    )
    t_c = 10.0 * torch.rand(200, 1)
    t_c.requires_grad_(True)
    t_0 = torch.zeros(1, 1, requires_grad=True)
    ottimizzatore = torch.optim.Adam(rete.parameters(), lr=1e-3)

    for epoca in range(epoche):
        ottimizzatore.zero_grad()
        u = rete(t_c)
        u_t = torch.autograd.grad(u, t_c, torch.ones_like(u),
                                  create_graph=True)[0]
        u_tt = torch.autograd.grad(u_t, t_c, torch.ones_like(u_t),
                                   create_graph=True)[0]
        loss_fisica = ((m * u_tt + c * u_t + k * u) ** 2).mean()

        u_0 = rete(t_0)
        u_t0 = torch.autograd.grad(u_0, t_0, torch.ones_like(u_0),
                                   create_graph=True)[0]
        loss_iniziale = (u_0 - 1.0).pow(2).mean() + u_t0.pow(2).mean()

        (loss_fisica + peso * loss_iniziale).backward()
        ottimizzatore.step()

        # residuo e errore vero, fianco a fianco
        if verboso and epoca % 2_500 == 0:
            errore_ora = diagnosi(rete, t_c)[2].max()
            print(f"epoca {epoca:6d} | residuo {loss_fisica.item():.2e}"
                  f" | errore vero {errore_ora:.3f}")

    return rete, t_c


rete_7, t_c7 = addestra(seme=7)
res_punti_7, res_griglia_7, errore_7 = diagnosi(rete_7, t_c7)

print(f"\n{'':<24}{'seme 42':>10}{'seme 7':>12}")
print(f"{'residuo sui suoi punti':<24}{res_punti:>10.2e}{res_punti_7:>12.2e}")
print(f"{'residuo sulla griglia':<24}{res_griglia:>10.2e}{res_griglia_7:>12.2e}")
print(f"{'errore vero':<24}{errore.max():>10.3f}{errore_7.max():>12.3f}")

# Le tre affermazioni che i numeri qui sopra devono reggere:
assert res_punti_7 < res_punti, (
    "il seme 7 non ha piu' il residuo piu' basso dei due."
)
assert errore_7.max() > errore.max(), (
    "il seme 7 non collassa piu' su questa versione di PyTorch."
)
assert res_griglia_7 > res_griglia, (
    "il residuo fuori dai punti di collocazione non e' piu' quello alto."
)

In [ ]:

def attraversamenti(rete):
    """Quante volte la curva della rete passa per lo zero in dieci secondi."""
    t = torch.tensor(t_test, dtype=torch.float32).reshape(-1, 1)
    with torch.no_grad():
        u = rete(t).squeeze().numpy()
    return int((np.sign(u[1:]) != np.sign(u[:-1])).sum())

risultati = {42: (rete, t_c), 7: (rete_7, t_c7)}
for seme in (0, 1, 2, 3):
    risultati[seme] = addestra(seme, verboso=False)

righe = []
for seme, (r, tc) in risultati.items():
    res, _, err = diagnosi(r, tc)
    righe.append((res, seme, err.max(), attraversamenti(r)))
print("seme   residuo   errore   zeri")
for res, seme, err, zeri in sorted(righe):
    print(f"{seme:4d}   {res:.1e}    {err:.3f}   {zeri:4d}")

# il moltiplicatore a 1, sullo stesso seme 42
rete_1, t_c1 = addestra(seme=42, peso=1.0, verboso=False)
res_punti_1, res_griglia_1, errore_1 = diagnosi(rete_1, t_c1)
with torch.no_grad():
    u0_1 = rete_1(torch.zeros(1, 1)).item()
print(f"\npeso 1: residuo sui punti {res_punti_1:.2e}, "
      f"sulla griglia {res_griglia_1:.2e}")
print(f"        errore vero {errore_1.max():.3f}, u(0) = {u0_1:.4f}")

def rapporto_gradienti(seme):
    """Ampiezza media dei gradienti della fisica contro quelli delle
    condizioni iniziali, sulla rete appena inizializzata."""
    torch.manual_seed(seme)
    r = nn.Sequential(
        nn.Linear(1, 32), nn.Tanh(),
        nn.Linear(32, 32), nn.Tanh(),
        nn.Linear(32, 32), nn.Tanh(),
        nn.Linear(32, 1),
    )
    tc = 10.0 * torch.rand(200, 1)
    tc.requires_grad_(True)
    t0 = torch.zeros(1, 1, requires_grad=True)
    u = r(tc)
    u_t = torch.autograd.grad(u, tc, torch.ones_like(u), create_graph=True)[0]
    u_tt = torch.autograd.grad(u_t, tc, torch.ones_like(u_t),
                               create_graph=True)[0]
    fisica = ((m * u_tt + c * u_t + k * u) ** 2).mean()
    u0 = r(t0)
    u_t0 = torch.autograd.grad(u0, t0, torch.ones_like(u0),
                               create_graph=True)[0]
    iniziale = (u0 - 1.0).pow(2).mean() + u_t0.pow(2).mean()
    pesi = list(r.parameters())
    g_f = torch.autograd.grad(fisica, pesi, retain_graph=True)
    g_i = torch.autograd.grad(iniziale, pesi)
    media = lambda gs: torch.cat([g.abs().flatten() for g in gs]).mean()
    return (media(g_f) / media(g_i)).item()

rapporti = np.array([rapporto_gradienti(s) for s in range(20)])
print(f"\nrapporto fra i gradienti all'avvio: "
      f"mediana {np.median(rapporti):.1f}, "
      f"rovesciato in {(rapporti < 1).sum()} semi su 20")

### Il problema inverso, in tre righe di codice


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

# k non lo conosciamo piu': diventa un parametro da apprendere
k_appreso = nn.Parameter(torch.tensor(1.0))   # partenza volutamente sbagliata

ottimizzatore = torch.optim.Adam(
    list(rete.parameters()) + [k_appreso], lr=1e-3
)

# nel ciclo di addestramento: il residuo usa il k appreso...
residuo = m * u_tt + c * u_t + k_appreso * u
# ...e accanto alla fisica c'e' il termine dati sulle misure rumorose
loss_dati = ((rete(t_oss) - u_oss) ** 2).mean()
loss = loss_fisica + 100.0 * loss_dati
```


In [ ]:

def problema_inverso(seme, epoche=30_000):
    """La molla nella scatola chiusa: k si impara insieme alla curva."""
    torch.manual_seed(seme)
    rete = nn.Sequential(
        nn.Linear(1, 32), nn.Tanh(),
        nn.Linear(32, 32), nn.Tanh(),
        nn.Linear(32, 32), nn.Tanh(),
        nn.Linear(32, 1),
    )
    t_c = 10.0 * torch.rand(200, 1)
    t_c.requires_grad_(True)
    # 25 misure equispaziate: la formula esatta con k = 4, piu' rumore
    t_oss = torch.linspace(0.0, 10.0, 25).reshape(-1, 1)
    t_np = t_oss.numpy()
    u_vera = torch.tensor(np.exp(-gamma * t_np) * (
        np.cos(omega_d * t_np) + (gamma / omega_d) * np.sin(omega_d * t_np)
    ), dtype=torch.float32)
    u_oss = u_vera + 0.05 * torch.randn(25, 1)

    k_appreso = nn.Parameter(torch.tensor(1.0))
    ottimizzatore = torch.optim.Adam(
        list(rete.parameters()) + [k_appreso], lr=1e-3
    )
    for _ in range(epoche):
        ottimizzatore.zero_grad()
        u = rete(t_c)
        u_t = torch.autograd.grad(u, t_c, torch.ones_like(u),
                                  create_graph=True)[0]
        u_tt = torch.autograd.grad(u_t, t_c, torch.ones_like(u_t),
                                   create_graph=True)[0]
        loss_fisica = ((m * u_tt + c * u_t + k_appreso * u) ** 2).mean()
        loss_dati = ((rete(t_oss) - u_oss) ** 2).mean()
        (loss_fisica + 100.0 * loss_dati).backward()
        ottimizzatore.step()

    t_griglia = torch.tensor(t_test, dtype=torch.float32).reshape(-1, 1)
    with torch.no_grad():
        errore = np.abs(rete(t_griglia).squeeze().numpy() - u_esatta).max()
    rumore = (u_oss - u_vera).abs().max().item()
    return k_appreso.item(), errore, rumore

for seme in (42, 7):
    k_stimato, errore_inv, rumore = problema_inverso(seme)
    print(f"seme {seme:2d}: k stimato {k_stimato:.2f}, scarto massimo dalla "
          f"curva vera {errore_inv:.3f}, misura peggiore {rumore:.3f}")

## Dove la fisica aiuta e dove no

[Leggi la pagina](https://book.paithon.it/main/PINN/applicazioni-limiti.html)


### Il problema inverso, cioè il superpotere


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

# Il parametro fisico ignoto (qui la diffusivita', cioe' quanto in fretta
# il calore si propaga nel materiale) diventa una manopola addestrabile,
# indistinguibile da un peso qualsiasi della rete. `rete` e' la candidata
# soluzione della sezione precedente.
alpha = torch.nn.Parameter(torch.tensor(0.5))          # valore iniziale di comodo
ottimizzatore = torch.optim.Adam(                      # ottimizzato insieme ai pesi
    list(rete.parameters()) + [alpha], lr=1e-3
)

# Stare nella lista non basta: alpha si muove solo se compare nel conto da
# cui il gradiente parte, cioe' dentro il residuo.
residuo = u_t - alpha * u_xx
```
